# Full 114M-Name SapBERT Contrastive Fine-Tuning V2

This notebook fine-tunes SapBERT on the full reference name database without loading the 114M rows into memory at once. V2 adds the upgrades missing from the first full-114M run: start from the best clean checkpoint, apply light training-quality filtering, build dynamic per-CID groups, use harder in-batch negatives, sample multiple positives per CID, and select the best checkpoint by validation retrieval MRR.

The important design choice is that we do **not** materialize every synonym pair. For large synonym sets, all possible pairs grow combinatorially. Instead, the notebook samples a fixed number of normalized names per CID from the full database, then dynamically samples names from each CID group during training.


## Install Dependencies

Run this first in Colab. It avoids reinstalling PyTorch so Colab keeps the CUDA build that matches the runtime.


In [ ]:

import subprocess
import sys

packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'duckdb>=1.0',
    'tqdm>=4.66',
    'PyYAML>=6.0',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
    'matplotlib>=3.8',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Installed notebook dependencies. If Colab asks for a restart after dependency changes, restart and continue from the next cell.')


## Mount Drive and Locate Repo


In [ ]:

from pathlib import Path
import os
import shutil
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer. Update find_repo_dir().')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration

The defaults continue from the smaller SapBERT contrastive fine-tune, but use a longer and gentler V2 continuation: 20,000 total training steps, a lower learning rate, validation-driven checkpoint selection, and checkpoint resume support. `NAMES_PER_CID = 4` gives each CID more pair variety without materializing all possible pairs.


In [ ]:

import gc
import json
import math
import re
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

from chem_synonymizer.normalize import normalize_name
from chem_synonymizer.utils import save_json, set_seed

# Input data. Update only if your Drive path differs.
REFERENCE_CSV_PATH = REPO_DIR / 'data' / 'CID-Synonym-full.csv'
CID_COL = 'cid'
NAME_COL = 'name'

# Model and output locations. Start from the best clean tuned model when present.
BEST_CLEAN_MODEL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'
BASE_MODEL_NAME = str(BEST_CLEAN_MODEL_DIR) if (BEST_CLEAN_MODEL_DIR / 'config.json').exists() else 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
RUN_NAME = 'sapbert_full_114m_contrastive_V2'
RESULTS_RUN_DIR = REPO_DIR / 'results' / RUN_NAME
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'
MODEL_BEST_DIR = RESULTS_ARTIFACTS_DIR / 'model_best'
MODEL_BEST_TRAINED_DIR = RESULTS_ARTIFACTS_DIR / 'model_best_trained'
MODEL_LAST_DIR = RESULTS_ARTIFACTS_DIR / 'model_last'
CHECKPOINT_ROOT = RESULTS_ARTIFACTS_DIR / 'checkpoints'
PLOTS_DIR = RESULTS_RUN_DIR / 'plots'
HISTORY_CSV_PATH = RESULTS_RUN_DIR / 'full114m_v2_finetune_history.csv'

# Use local scratch for the heavy DuckDB and pair-building work.
SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').exists() else Path('/content')
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / RUN_NAME
WORK_ARTIFACTS_DIR = WORK_DIR / 'artifacts'
PREPARE_DB_PATH = WORK_ARTIFACTS_DIR / 'full114m_v2_group_prep.duckdb'
TRAIN_GROUPS_PATH = WORK_ARTIFACTS_DIR / 'train_cid_groups_v2.parquet'
VAL_GROUPS_PATH = WORK_ARTIFACTS_DIR / 'val_cid_groups_v2.parquet'
GROUP_STATS_PATH = WORK_ARTIFACTS_DIR / 'group_stats_v2.json'

# Data preparation.
SEED = 0
CPU_THREADS = max(1, os.cpu_count() or 8)
REFERENCE_CHUNKSIZE = 2_000_000
NAMES_PER_CID = 4
VAL_FRAC_BY_CID = 0.02
VALIDATION_GROUP_LIMIT = 50_000
REBUILD_TRAINING_GROUPS = False
DELETE_PREPARE_DB_AFTER_PARQUET = True
MIN_TRAIN_NAME_LEN = 3
MAX_TRAIN_NAME_LEN = 180
MIN_COMPACT_LEN = 3
FILTER_SOURCE_TAG_SUFFIXES = True
FILTER_GENERIC_SHORT_ALPHA = True
USE_HARD_NEGATIVE_BATCHING = True
HARD_KEY_PREFIX_LEN = 8
HARD_KEY_LENGTH_BIN = 8

# Training. One batch group contributes two texts sampled from the same CID, matching SynonymBatchFactory.contrastive_batch.
MAX_LENGTH = 96
POOLING = 'cls'
TRAIN_GROUP_BATCH_SIZE = 128
NAMES_PER_GROUP_IN_BATCH = 3
MAX_TRAIN_STEPS = 20_000
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
TEMPERATURE = 0.05
MIXED_PRECISION = 'bf16'  # A100-friendly. Use 'fp16' if bf16 is unavailable.
SAVE_EVERY_STEPS = 1000
VALIDATE_EVERY_STEPS = 500
VALIDATION_PAIR_LIMIT = 8192
VALIDATION_ENCODE_BATCH_SIZE = 512
VALIDATION_SCORE_BATCH_SIZE = 512
KEEP_LAST_N_CHECKPOINTS = 4
OVERWRITE_MODEL = False
RESUME_FROM_CHECKPOINT = True

for path in [RESULTS_RUN_DIR, RESULTS_ARTIFACTS_DIR, CHECKPOINT_ROOT, WORK_ARTIFACTS_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not REFERENCE_CSV_PATH.exists():
    raise FileNotFoundError(REFERENCE_CSV_PATH)

set_seed(SEED)
print('Reference CSV:', REFERENCE_CSV_PATH)
print('Start model:', BASE_MODEL_NAME)
print('Results:', RESULTS_RUN_DIR)
print('Scratch:', WORK_DIR, shutil.disk_usage(SCRATCH_ROOT))
print('CPU threads:', CPU_THREADS)


## Helpers


In [ ]:

def normalize_series(values: pd.Series) -> pd.Series:
    # Vectorized equivalent of chem_synonymizer.normalize.normalize_name.
    return (
        values.fillna('')
        .astype('string')
        .str.normalize('NFKC')
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )


def training_quality_mask(names: pd.Series) -> pd.Series:
    names = names.fillna('').astype('string')
    compact = names.str.replace(r'[^a-z0-9]+', '', regex=True)
    mask = names.str.len().between(MIN_TRAIN_NAME_LEN, MAX_TRAIN_NAME_LEN)
    mask &= compact.str.len() >= MIN_COMPACT_LEN
    mask &= compact.str.contains(r'[a-z]', regex=True, na=False)
    if FILTER_SOURCE_TAG_SUFFIXES:
        mask &= ~names.str.contains(r'\[[a-z]{1,8}\]\s*$', regex=True, na=False)
    if FILTER_GENERIC_SHORT_ALPHA:
        mask &= ~names.str.fullmatch(r'[a-z]{1,3}', na=False)
    return mask.fillna(False)


def q(path: Path) -> str:
    return str(path).replace("'", "''")


def result_artifact_path(work_path: Path) -> Path:
    return RESULTS_ARTIFACTS_DIR / work_path.name


def restore_artifact(work_path: Path) -> None:
    result_path = result_artifact_path(work_path)
    if work_path.exists() or not result_path.exists():
        return
    work_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Restoring {result_path.name} to scratch...')
    shutil.copy2(result_path, work_path)


def backup_artifact(work_path: Path) -> None:
    if not work_path.exists():
        return
    result_path = result_artifact_path(work_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    if result_path.exists() and result_path.stat().st_size == work_path.stat().st_size:
        print('Drive artifact already backed up:', result_path.name)
        return
    if result_path.exists():
        result_path.unlink()
    print(f'Backing up {work_path.name} to Drive results...')
    shutil.copy2(work_path, result_path)
    if result_path.stat().st_size != work_path.stat().st_size:
        raise OSError(f'Backup size mismatch for {work_path.name}')


def save_config() -> None:
    payload = {
        'run_name': RUN_NAME,
        'reference_csv_path': str(REFERENCE_CSV_PATH),
        'start_model': BASE_MODEL_NAME,
        'normalization': 'NFKC + lowercase + strip + whitespace collapse',
        'training_quality_filter': {
            'min_train_name_len': MIN_TRAIN_NAME_LEN,
            'max_train_name_len': MAX_TRAIN_NAME_LEN,
            'min_compact_len': MIN_COMPACT_LEN,
            'filter_source_tag_suffixes': FILTER_SOURCE_TAG_SUFFIXES,
            'filter_generic_short_alpha': FILTER_GENERIC_SHORT_ALPHA,
        },
        'cid_col': CID_COL,
        'name_col': NAME_COL,
        'seed': SEED,
        'cpu_threads': CPU_THREADS,
        'reference_chunksize': REFERENCE_CHUNKSIZE,
        'names_per_cid': NAMES_PER_CID,
        'val_frac_by_cid': VAL_FRAC_BY_CID,
        'validation_group_limit': VALIDATION_GROUP_LIMIT,
        'use_hard_negative_batching': USE_HARD_NEGATIVE_BATCHING,
        'hard_key_prefix_len': HARD_KEY_PREFIX_LEN,
        'hard_key_length_bin': HARD_KEY_LENGTH_BIN,
        'max_length': MAX_LENGTH,
        'pooling': POOLING,
        'train_group_batch_size': TRAIN_GROUP_BATCH_SIZE,
        'names_per_group_in_batch': NAMES_PER_GROUP_IN_BATCH,
        'max_train_steps': MAX_TRAIN_STEPS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'max_grad_norm': MAX_GRAD_NORM,
        'temperature': TEMPERATURE,
        'mixed_precision': MIXED_PRECISION,
        'validate_every_steps': VALIDATE_EVERY_STEPS,
        'validation_pair_limit': VALIDATION_PAIR_LIMIT,
        'model_best_dir': str(MODEL_BEST_DIR),
        'model_best_trained_dir': str(MODEL_BEST_TRAINED_DIR),
        'model_last_dir': str(MODEL_LAST_DIR),
    }
    save_json(payload, RESULTS_RUN_DIR / 'full114m_finetune_config.json')


save_config()


## Build Full-Database CID Groups

This stage streams `CID-Synonym-full.csv`, normalizes names, removes duplicate `(cid, normalized_name)` pairs, keeps CIDs with at least two normalized names, samples `NAMES_PER_CID` names per CID, and writes one row per CID group to Parquet.

During training, each batch samples two names from each CID group dynamically. This matches the smaller `SynonymBatchFactory.contrastive_batch` pattern more closely than the previous fixed-pair version.


In [ ]:
def prepare_training_groups_v2() -> dict:
    restore_artifact(TRAIN_GROUPS_PATH)
    restore_artifact(VAL_GROUPS_PATH)
    restore_artifact(GROUP_STATS_PATH)
    if TRAIN_GROUPS_PATH.exists() and VAL_GROUPS_PATH.exists() and GROUP_STATS_PATH.exists() and not REBUILD_TRAINING_GROUPS:
        with GROUP_STATS_PATH.open('r', encoding='utf-8') as f:
            stats = json.load(f)
        print('Using cached V2 CID groups.')
        print(json.dumps(stats, indent=2)[:3000])
        return stats

    for path in [TRAIN_GROUPS_PATH, VAL_GROUPS_PATH, GROUP_STATS_PATH, PREPARE_DB_PATH]:
        if path.exists() and path.is_file():
            path.unlink()

    con = duckdb.connect(str(PREPARE_DB_PATH))
    con.execute(f'PRAGMA threads={int(CPU_THREADS)}')
    con.execute('PRAGMA preserve_insertion_order=false')
    con.execute('DROP TABLE IF EXISTS reference_pairs')
    con.execute('CREATE TABLE reference_pairs (cid VARCHAR, normalized_name VARCHAR)')

    total_rows = 0
    quality_rows = 0
    inserted_rows = 0
    chunk_i = 0
    t0 = time.perf_counter()
    reader = pd.read_csv(
        REFERENCE_CSV_PATH,
        usecols=[CID_COL, NAME_COL],
        dtype={CID_COL: 'string', NAME_COL: 'string'},
        chunksize=REFERENCE_CHUNKSIZE,
        on_bad_lines='skip',
    )

    for chunk in tqdm(reader, desc='Streaming, normalizing, and filtering reference CSV'):
        chunk_i += 1
        total_rows += len(chunk)
        out = pd.DataFrame({
            'cid': chunk[CID_COL].astype('string'),
            'normalized_name': normalize_series(chunk[NAME_COL]),
        })
        out = out.dropna(subset=['cid'])
        out = out[out['normalized_name'].str.len() > 0]
        out = out[training_quality_mask(out['normalized_name'])]
        quality_rows += len(out)
        out = out.drop_duplicates(['cid', 'normalized_name'])
        inserted_rows += len(out)
        con.register('chunk_df', out)
        con.execute('INSERT INTO reference_pairs SELECT CAST(cid AS VARCHAR), CAST(normalized_name AS VARCHAR) FROM chunk_df')
        con.unregister('chunk_df')
        del chunk, out
        if chunk_i % 5 == 0:
            gc.collect()

    print('Creating global deduplicated training table...')
    con.execute('DROP TABLE IF EXISTS reference_dedup')
    con.execute('''
        CREATE TABLE reference_dedup AS
        SELECT DISTINCT cid, normalized_name
        FROM reference_pairs
        WHERE cid IS NOT NULL AND normalized_name IS NOT NULL AND length(normalized_name) > 0
    ''')
    con.execute('DROP TABLE reference_pairs')

    print('Counting trainable CIDs...')
    con.execute('DROP TABLE IF EXISTS cid_counts')
    con.execute('''
        CREATE TABLE cid_counts AS
        SELECT cid, COUNT(*) AS n_names
        FROM reference_dedup
        GROUP BY cid
        HAVING COUNT(*) >= 2
    ''')

    print(f'Sampling up to {NAMES_PER_CID} names per CID...')
    con.execute('DROP TABLE IF EXISTS sampled_names')
    con.execute(f'''
        CREATE TABLE sampled_names AS
        SELECT cid, normalized_name, rn
        FROM (
            SELECT
                d.cid,
                d.normalized_name,
                ROW_NUMBER() OVER (
                    PARTITION BY d.cid
                    ORDER BY hash(d.normalized_name || '|name|' || CAST({int(SEED)} AS VARCHAR))
                ) AS rn
            FROM reference_dedup d
            INNER JOIN cid_counts c USING (cid)
        )
        WHERE rn <= {int(NAMES_PER_CID)}
    ''')

    print('Building grouped CID table with hard-negative keys...')
    con.execute('DROP TABLE IF EXISTS cid_groups')
    con.execute(f'''
        CREATE TABLE cid_groups AS
        WITH grouped AS (
            SELECT
                cid,
                list(normalized_name ORDER BY rn) AS names,
                COUNT(*) AS n_names,
                min(normalized_name) AS key_name
            FROM sampled_names
            GROUP BY cid
            HAVING COUNT(*) >= 2
        ), keyed AS (
            SELECT
                cid,
                names,
                n_names,
                substr(regexp_replace(lower(key_name), '[^a-z0-9]+', '', 'g'), 1, {int(HARD_KEY_PREFIX_LEN)})
                    || '_' ||
                CAST(LEAST(CAST(length(regexp_replace(lower(key_name), '[^a-z0-9]+', '', 'g')) / {int(HARD_KEY_LENGTH_BIN)} AS INTEGER), 32) AS VARCHAR)
                    AS hard_key,
                hash(cid || '|split|' || CAST({int(SEED)} AS VARCHAR)) % 1000000 AS split_hash
            FROM grouped
        )
        SELECT * FROM keyed
    ''')

    val_threshold = int(float(VAL_FRAC_BY_CID) * 1_000_000)
    if USE_HARD_NEGATIVE_BATCHING:
        order_clause = "hard_key, hash(cid || '|train|' || CAST({} AS VARCHAR))".format(int(SEED))
    else:
        order_clause = "hash(cid || '|train|' || CAST({} AS VARCHAR))".format(int(SEED))

    print('Writing validation CID groups...')
    con.execute(f'''
        COPY (
            SELECT cid, names, n_names, hard_key
            FROM cid_groups
            WHERE split_hash < {val_threshold}
            ORDER BY hard_key, hash(cid || '|val|' || CAST({int(SEED)} AS VARCHAR))
            LIMIT {int(VALIDATION_GROUP_LIMIT)}
        ) TO '{q(VAL_GROUPS_PATH)}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    ''')

    print('Writing train CID groups...')
    con.execute(f'''
        COPY (
            SELECT cid, names, n_names, hard_key
            FROM cid_groups
            WHERE split_hash >= {val_threshold}
            ORDER BY {order_clause}
        ) TO '{q(TRAIN_GROUPS_PATH)}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    ''')

    stats = {
        'total_csv_rows_seen': int(total_rows),
        'quality_filtered_rows': int(quality_rows),
        'chunk_dedup_rows_inserted': int(inserted_rows),
        'unique_training_cid_name_pairs': int(con.execute('SELECT COUNT(*) FROM reference_dedup').fetchone()[0]),
        'trainable_cids_after_filter': int(con.execute('SELECT COUNT(*) FROM cid_counts').fetchone()[0]),
        'sampled_name_rows': int(con.execute('SELECT COUNT(*) FROM sampled_names').fetchone()[0]),
        'train_groups': int(con.execute(f"SELECT COUNT(*) FROM read_parquet('{q(TRAIN_GROUPS_PATH)}')").fetchone()[0]),
        'validation_groups': int(con.execute(f"SELECT COUNT(*) FROM read_parquet('{q(VAL_GROUPS_PATH)}')").fetchone()[0]),
        'names_per_cid': int(NAMES_PER_CID),
        'use_hard_negative_batching': bool(USE_HARD_NEGATIVE_BATCHING),
        'elapsed_sec': float(time.perf_counter() - t0),
    }
    save_json(stats, GROUP_STATS_PATH)
    con.close()

    if DELETE_PREPARE_DB_AFTER_PARQUET and PREPARE_DB_PATH.exists():
        PREPARE_DB_PATH.unlink()

    backup_artifact(TRAIN_GROUPS_PATH)
    backup_artifact(VAL_GROUPS_PATH)
    backup_artifact(GROUP_STATS_PATH)
    print(json.dumps(stats, indent=2))
    return stats


group_stats = prepare_training_groups_v2()


## Fine-Tune SapBERT with Supervised Contrastive Loss

This cell resumes from the latest checkpoint when one exists. It only skips training when the latest checkpoint has already reached `MAX_TRAIN_STEPS`; an existing `model_best` is preserved and only replaced when a later validation run improves MRR.


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from chem_synonymizer.embed import pool_outputs


def latest_checkpoint() -> Path | None:
    checkpoints = []
    for path in CHECKPOINT_ROOT.glob('checkpoint-*'):
        try:
            step = int(path.name.split('-')[-1])
        except ValueError:
            continue
        if (path / 'config.json').exists():
            checkpoints.append((step, path))
    if not checkpoints:
        return None
    return sorted(checkpoints)[-1][1]


def checkpoint_step(path: Path | None) -> int:
    if path is None:
        return 0
    try:
        return int(path.name.split('-')[-1])
    except ValueError:
        return 0


def cleanup_old_checkpoints() -> None:
    checkpoints = []
    for path in CHECKPOINT_ROOT.glob('checkpoint-*'):
        try:
            step = int(path.name.split('-')[-1])
        except ValueError:
            continue
        checkpoints.append((step, path))
    for _, path in sorted(checkpoints)[:-int(KEEP_LAST_N_CHECKPOINTS)]:
        shutil.rmtree(path, ignore_errors=True)


def load_training_history() -> list[dict]:
    if not HISTORY_CSV_PATH.exists():
        return []
    return pd.read_csv(HISTORY_CSV_PATH).to_dict('records')


def write_training_history(history: list[dict]) -> None:
    if history:
        df = pd.DataFrame(history)
        dedupe_cols = [col for col in ['step', 'validation_MRR', 'loss'] if col in df.columns]
        if dedupe_cols:
            df = df.drop_duplicates(subset=dedupe_cols, keep='last')
        df.to_csv(HISTORY_CSV_PATH, index=False)


def load_existing_best() -> tuple[float, int | None]:
    best_path = RESULTS_RUN_DIR / 'best_validation_metrics.json'
    if not best_path.exists():
        return float('-inf'), None
    payload = json.loads(best_path.read_text(encoding='utf-8'))
    metrics = payload.get('validation_metrics', {})
    mrr = metrics.get('MRR', float('-inf'))
    return float(mrr), payload.get('best_step')


def load_existing_best_trained() -> tuple[float, int | None]:
    best_path = RESULTS_RUN_DIR / 'best_trained_validation_metrics.json'
    if not best_path.exists():
        return float('-inf'), None
    payload = json.loads(best_path.read_text(encoding='utf-8'))
    metrics = payload.get('validation_metrics', {})
    mrr = metrics.get('MRR', float('-inf'))
    return float(mrr), payload.get('best_trained_step')


def set_scheduler_resume_step(scheduler, optimizer, completed_steps: int) -> None:
    # Avoid calling scheduler.step() before optimizer.step() while still resuming
    # the LR schedule at the number of already-completed optimizer updates.
    completed_steps = int(completed_steps)
    if completed_steps <= 0:
        return
    scheduler.last_epoch = completed_steps
    if not hasattr(scheduler, 'lr_lambdas'):
        return
    lrs = [
        float(base_lr) * float(lr_lambda(completed_steps))
        for base_lr, lr_lambda in zip(scheduler.base_lrs, scheduler.lr_lambdas)
    ]
    for group, lr in zip(optimizer.param_groups, lrs):
        group['lr'] = lr
    scheduler._last_lr = lrs


def supervised_contrastive_loss(embeddings: torch.Tensor, labels: torch.Tensor, temperature: float) -> torch.Tensor:
    labels = labels.to(embeddings.device)
    sim = torch.matmul(embeddings, embeddings.T) / float(temperature)
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    eye = torch.eye(labels.numel(), dtype=torch.bool, device=embeddings.device)
    positive_mask = labels[:, None].eq(labels[None, :]) & ~eye
    logits_mask = ~eye
    exp_logits = torch.exp(sim) * logits_mask.float()
    log_prob = sim - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp_min(1e-12))
    positives_per_anchor = positive_mask.sum(dim=1)
    valid = positives_per_anchor > 0
    mean_log_prob = (positive_mask.float() * log_prob).sum(dim=1) / positives_per_anchor.clamp_min(1)
    return -mean_log_prob[valid].mean()


def encode_train_batch(texts: list[str], tokenizer, model, device: str) -> torch.Tensor:
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=int(MAX_LENGTH),
        return_tensors='pt',
    ).to(device)
    if MIXED_PRECISION == 'bf16':
        amp_dtype = torch.bfloat16
        amp_enabled = device.startswith('cuda')
    elif MIXED_PRECISION == 'fp16':
        amp_dtype = torch.float16
        amp_enabled = device.startswith('cuda')
    else:
        amp_dtype = torch.float32
        amp_enabled = False
    with torch.autocast(device_type='cuda' if device.startswith('cuda') else 'cpu', dtype=amp_dtype, enabled=amp_enabled):
        out = model(**enc)
        emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    del enc, out
    return emb


def dynamic_contrastive_batch_iterator(parquet_path: Path, batch_groups: int, names_per_group: int, seed: int, skip_batches: int = 0):
    pf = pq.ParquetFile(parquet_path)
    rng = np.random.default_rng(int(seed) + int(skip_batches))
    seen_batches = 0
    while True:
        for batch in pf.iter_batches(batch_size=int(batch_groups), columns=['cid', 'names', 'hard_key']):
            if seen_batches < skip_batches:
                seen_batches += 1
                continue
            df = batch.to_pandas().dropna(subset=['cid', 'names'])
            texts = []
            labels = []
            kept_groups = 0
            for names in df['names'].tolist():
                names = [str(name) for name in list(names) if str(name)]
                if len(names) < 2:
                    continue
                sample_n = min(int(names_per_group), len(names))
                chosen = rng.choice(len(names), size=sample_n, replace=False)
                texts.extend([names[int(i)] for i in chosen])
                labels.extend([kept_groups] * sample_n)
                kept_groups += 1
            seen_batches += 1
            if texts:
                yield texts, torch.tensor(labels, dtype=torch.long)


def validation_pairs_from_groups(limit: int, seed: int) -> tuple[list[str], list[str]]:
    table = pq.read_table(VAL_GROUPS_PATH, columns=['names']).slice(0, int(limit))
    df = table.to_pandas()
    rng = np.random.default_rng(int(seed))
    anchors = []
    positives = []
    for names in df['names'].tolist():
        names = [str(name) for name in list(names) if str(name)]
        if len(names) < 2:
            continue
        a_i, p_i = rng.choice(len(names), size=2, replace=False)
        anchors.append(names[int(a_i)])
        positives.append(names[int(p_i)])
    return anchors, positives


def encode_texts_for_eval(texts: list[str], tokenizer, model, device: str, batch_size: int) -> torch.Tensor:
    chunks = []
    was_training = model.training
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(texts), int(batch_size)):
            batch = texts[start:start + int(batch_size)]
            emb = encode_train_batch(batch, tokenizer, model, device)
            chunks.append(emb.detach().cpu())
    if was_training:
        model.train()
    return torch.cat(chunks, dim=0)


def evaluate_validation_retrieval(tokenizer, model, device: str, limit: int = VALIDATION_PAIR_LIMIT) -> dict:
    anchors, positives = validation_pairs_from_groups(limit, seed=SEED)
    n = len(anchors)
    if n == 0:
        raise ValueError('No validation pairs available.')
    anchor_emb = encode_texts_for_eval(anchors, tokenizer, model, device, VALIDATION_ENCODE_BATCH_SIZE).to(device)
    positive_emb = encode_texts_for_eval(positives, tokenizer, model, device, VALIDATION_ENCODE_BATCH_SIZE).to(device)
    ranks = []
    with torch.inference_mode():
        for start in range(0, n, int(VALIDATION_SCORE_BATCH_SIZE)):
            end = min(start + int(VALIDATION_SCORE_BATCH_SIZE), n)
            scores = anchor_emb[start:end] @ positive_emb.T
            target = scores[torch.arange(end - start, device=device), torch.arange(start, end, device=device)]
            rank = (scores > target[:, None]).sum(dim=1) + 1
            ranks.append(rank.detach().cpu())
            del scores, target, rank
    ranks = torch.cat(ranks).numpy()
    payload = {
        'n_pairs': int(n),
        'MRR': float(np.mean(1.0 / ranks)),
        'HitRate@1': float(np.mean(ranks <= 1)),
        'HitRate@5': float(np.mean(ranks <= 5)),
        'HitRate@10': float(np.mean(ranks <= 10)),
        'median_rank': float(np.median(ranks)),
    }
    del anchor_emb, positive_emb
    if device.startswith('cuda'):
        torch.cuda.empty_cache()
    return payload


def save_training_checkpoint(model, tokenizer, step: int, history: list[dict]) -> None:
    ckpt_dir = CHECKPOINT_ROOT / f'checkpoint-{int(step):06d}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    save_json({'step': int(step), 'history_tail': history[-10:]}, ckpt_dir / 'training_state.json')
    write_training_history(history)
    cleanup_old_checkpoints()


def save_best_model(model, tokenizer, step: int, metrics: dict) -> None:
    MODEL_BEST_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_BEST_DIR)
    tokenizer.save_pretrained(MODEL_BEST_DIR)
    save_json({'best_step': int(step), 'validation_metrics': metrics}, RESULTS_RUN_DIR / 'best_validation_metrics.json')


def save_best_trained_model(model, tokenizer, step: int, metrics: dict) -> None:
    MODEL_BEST_TRAINED_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_BEST_TRAINED_DIR)
    tokenizer.save_pretrained(MODEL_BEST_TRAINED_DIR)
    save_json({'best_trained_step': int(step), 'validation_metrics': metrics}, RESULTS_RUN_DIR / 'best_trained_validation_metrics.json')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device.startswith('cuda'):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
print('Training device:', device)

target_steps = int(MAX_TRAIN_STEPS)
resume_checkpoint = latest_checkpoint() if RESUME_FROM_CHECKPOINT else None
resume_step = checkpoint_step(resume_checkpoint)

if (resume_step >= target_steps) and (MODEL_BEST_DIR / 'config.json').exists() and not OVERWRITE_MODEL:
    print(f'Training already reached {resume_step} steps; skipping training. Increase MAX_TRAIN_STEPS to continue.')
else:
    start_step = 0
    model_source = BASE_MODEL_NAME
    ckpt = resume_checkpoint
    if ckpt is not None and not OVERWRITE_MODEL:
        model_source = str(ckpt)
        start_step = checkpoint_step(ckpt)
        print('Resuming model weights from checkpoint:', ckpt)
    else:
        print('Starting from:', BASE_MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(model_source, use_fast=True)
    model = AutoModel.from_pretrained(model_source).to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=float(LEARNING_RATE), weight_decay=float(WEIGHT_DECAY))
    total_steps = int(MAX_TRAIN_STEPS)
    warmup_steps = int(total_steps * float(WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    set_scheduler_resume_step(scheduler, optimizer, start_step)

    history = load_training_history() if start_step > 0 and not OVERWRITE_MODEL else []
    running_loss = None
    iterator = dynamic_contrastive_batch_iterator(
        TRAIN_GROUPS_PATH,
        TRAIN_GROUP_BATCH_SIZE,
        NAMES_PER_GROUP_IN_BATCH,
        seed=SEED,
        skip_batches=start_step,
    )
    t0 = time.perf_counter()

    initial_metrics = evaluate_validation_retrieval(tokenizer, model, device)
    existing_best_mrr, existing_best_step = load_existing_best()
    existing_best_trained_mrr, existing_best_trained_step = load_existing_best_trained()
    best_mrr = existing_best_mrr
    best_step = existing_best_step
    best_trained_mrr = existing_best_trained_mrr
    best_trained_step = existing_best_trained_step
    initial_row = {
        'step': int(start_step),
        'validation_MRR': initial_metrics['MRR'],
        'validation_HitRate@1': initial_metrics['HitRate@1'],
        'validation_HitRate@5': initial_metrics['HitRate@5'],
        'validation_HitRate@10': initial_metrics['HitRate@10'],
        'validation_median_rank': initial_metrics['median_rank'],
    }
    history.append(initial_row)
    write_training_history(history)
    if OVERWRITE_MODEL or initial_metrics['MRR'] > best_mrr or not (MODEL_BEST_DIR / 'config.json').exists():
        best_mrr = initial_metrics['MRR']
        best_step = start_step
        save_best_model(model, tokenizer, start_step, initial_metrics)
    if start_step > 0 and (OVERWRITE_MODEL or initial_metrics['MRR'] > best_trained_mrr or not (MODEL_BEST_TRAINED_DIR / 'config.json').exists()):
        best_trained_mrr = initial_metrics['MRR']
        best_trained_step = start_step
        save_best_trained_model(model, tokenizer, start_step, initial_metrics)
    print('Initial validation:', json.dumps(initial_metrics, indent=2))
    print(f'Best overall validation MRR to beat: {best_mrr:.4f} at step {best_step}')
    print(f'Best trained validation MRR so far: {best_trained_mrr:.4f} at step {best_trained_step}')

    progress = tqdm(range(start_step, total_steps), desc='Fine-tuning SapBERT V2')
    for step in progress:
        texts, labels = next(iterator)
        optimizer.zero_grad(set_to_none=True)
        embeddings = encode_train_batch(texts, tokenizer, model, device)
        loss = supervised_contrastive_loss(embeddings, labels, TEMPERATURE)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(MAX_GRAD_NORM))
        optimizer.step()
        scheduler.step()

        loss_value = float(loss.detach().cpu())
        running_loss = loss_value if running_loss is None else (0.95 * running_loss + 0.05 * loss_value)
        progress.set_postfix(loss=f'{running_loss:.4f}', best_mrr=f'{best_mrr:.4f}')

        if ((step + 1) % 100 == 0) or (step + 1 == total_steps):
            history.append({
                'step': int(step + 1),
                'loss': loss_value,
                'running_loss': float(running_loss),
                'elapsed_sec': float(time.perf_counter() - t0),
            })

        if ((step + 1) % int(VALIDATE_EVERY_STEPS) == 0) or (step + 1 == total_steps):
            metrics = evaluate_validation_retrieval(tokenizer, model, device)
            metrics_row = {
                'step': int(step + 1),
                'validation_MRR': metrics['MRR'],
                'validation_HitRate@1': metrics['HitRate@1'],
                'validation_HitRate@5': metrics['HitRate@5'],
                'validation_HitRate@10': metrics['HitRate@10'],
                'validation_median_rank': metrics['median_rank'],
            }
            history.append(metrics_row)
            write_training_history(history)
            print('Validation:', json.dumps(metrics_row, indent=2))
            if metrics['MRR'] > best_mrr:
                best_mrr = metrics['MRR']
                best_step = step + 1
                save_best_model(model, tokenizer, step + 1, metrics)
            if metrics['MRR'] > best_trained_mrr:
                best_trained_mrr = metrics['MRR']
                best_trained_step = step + 1
                save_best_trained_model(model, tokenizer, step + 1, metrics)

        if ((step + 1) % int(SAVE_EVERY_STEPS) == 0) or (step + 1 == total_steps):
            save_training_checkpoint(model, tokenizer, step + 1, history)

        del texts, labels, embeddings, loss
        if device.startswith('cuda') and (step + 1) % 25 == 0:
            torch.cuda.empty_cache()

    MODEL_LAST_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MODEL_LAST_DIR)
    tokenizer.save_pretrained(MODEL_LAST_DIR)
    write_training_history(history)
    save_json(
        {
            'run_name': RUN_NAME,
            'start_model': BASE_MODEL_NAME,
            'best_model_dir': str(MODEL_BEST_DIR),
            'best_trained_model_dir': str(MODEL_BEST_TRAINED_DIR),
            'last_model_dir': str(MODEL_LAST_DIR),
            'best_step': int(best_step) if best_step is not None else None,
            'best_validation_MRR': float(best_mrr),
            'best_trained_step': int(best_trained_step) if best_trained_step is not None else None,
            'best_trained_validation_MRR': float(best_trained_mrr),
            'group_stats': group_stats,
            'max_train_steps': int(MAX_TRAIN_STEPS),
            'elapsed_sec': float(time.perf_counter() - t0),
        },
        RESULTS_RUN_DIR / 'full114m_v2_finetune_metrics.json',
    )
    print('Saved best overall model:', MODEL_BEST_DIR)
    print('Saved best trained model:', MODEL_BEST_TRAINED_DIR)
    print('Saved last model:', MODEL_LAST_DIR)


## Training Curves and Stop/Continue Check

These cells read the saved history CSV, so they can be rerun after a reconnect without retraining. Use MRR as the primary checkpoint-selection metric; median rank is useful for seeing whether the wider ranking distribution is still improving.


In [ ]:
import matplotlib.pyplot as plt

if not HISTORY_CSV_PATH.exists():
    print('No training history found yet:', HISTORY_CSV_PATH)
else:
    history_df = pd.read_csv(HISTORY_CSV_PATH)
    val_df = (
        history_df.dropna(subset=['validation_MRR'])
        .sort_values('step')
        .drop_duplicates(subset=['step'], keep='last')
        .reset_index(drop=True)
    )
    if val_df.empty:
        print('Training history exists, but it has no validation rows yet.')
    else:
        best_idx = val_df['validation_MRR'].idxmax()
        best_row = val_df.loc[best_idx]
        latest_row = val_df.iloc[-1]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        axes[0].plot(val_df['step'], val_df['validation_MRR'], marker='o', linewidth=2)
        axes[0].scatter([best_row['step']], [best_row['validation_MRR']], s=80, zorder=3)
        axes[0].set_title('Validation MRR')
        axes[0].set_xlabel('Training step')
        axes[0].set_ylabel('MRR')
        axes[0].grid(alpha=0.25)

        axes[1].plot(val_df['step'], val_df['validation_median_rank'], marker='o', linewidth=2, color='tab:orange')
        axes[1].scatter([latest_row['step']], [latest_row['validation_median_rank']], s=80, zorder=3, color='tab:orange')
        axes[1].set_title('Validation Median Rank')
        axes[1].set_xlabel('Training step')
        axes[1].set_ylabel('Median rank, lower is better')
        axes[1].grid(alpha=0.25)

        fig.suptitle(f'{RUN_NAME}: validation retrieval trend', y=1.02)
        fig.tight_layout()
        plot_path = PLOTS_DIR / 'v2_validation_mrr_median_rank.png'
        fig.savefig(plot_path, dpi=200, bbox_inches='tight')
        plt.show()
        print('Saved plot:', plot_path)


In [ ]:
if not HISTORY_CSV_PATH.exists():
    print('No training history found yet.')
else:
    history_df = pd.read_csv(HISTORY_CSV_PATH)
    val_df = (
        history_df.dropna(subset=['validation_MRR'])
        .sort_values('step')
        .drop_duplicates(subset=['step'], keep='last')
        .reset_index(drop=True)
    )
    if len(val_df) < 2:
        print('Need at least two validation points before making a trend decision.')
    else:
        first = val_df.iloc[0]
        latest = val_df.iloc[-1]
        best = val_df.loc[val_df['validation_MRR'].idxmax()]
        recent = val_df.tail(min(4, len(val_df)))
        recent_mrr_delta = float(recent['validation_MRR'].iloc[-1] - recent['validation_MRR'].iloc[0])
        recent_rank_delta = float(recent['validation_median_rank'].iloc[-1] - recent['validation_median_rank'].iloc[0])

        print(f"Initial step {int(first['step'])}: MRR={first['validation_MRR']:.4f}, median_rank={first['validation_median_rank']:.1f}")
        print(f"Best step {int(best['step'])}: MRR={best['validation_MRR']:.4f}, median_rank={best['validation_median_rank']:.1f}")
        print(f"Latest step {int(latest['step'])}: MRR={latest['validation_MRR']:.4f}, median_rank={latest['validation_median_rank']:.1f}")
        print(f"Recent trend over {len(recent)} validation points: delta MRR={recent_mrr_delta:+.4f}, delta median_rank={recent_rank_delta:+.1f}")

        if latest['validation_MRR'] >= first['validation_MRR'] and recent_mrr_delta >= -0.001:
            recommendation = 'Continue or evaluate externally: MRR has recovered to at least the starting checkpoint.'
        elif recent_mrr_delta > 0 and recent_rank_delta < 0:
            recommendation = 'Continue training: MRR and median rank are moving in the right direction, but do not use this checkpoint externally until MRR beats the starting checkpoint.'
        elif recent_rank_delta < 0 and abs(recent_mrr_delta) <= 0.002:
            recommendation = 'Continue cautiously: median rank is improving, but MRR is not clearly better yet. Keep the lower learning rate and reassess after more validation points.'
        else:
            recommendation = 'Stop or lower the learning rate: recent validation is not improving enough to justify replacing the starting checkpoint.'
        print('Recommendation:', recommendation)


## Final Validation Check

This repeats validation retrieval for the saved best-overall checkpoint and the best V2-trained checkpoint when both exist. The external lab FAISS notebook is still the real test; this validation check is only for checkpoint selection during training.


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_specs = [
    ('best_overall_mrr', MODEL_BEST_DIR),
    ('best_trained_mrr', MODEL_BEST_TRAINED_DIR),
]

final_validation_payload = {}
for label, model_dir in checkpoint_specs:
    if not (model_dir / 'config.json').exists():
        print(f'Skipping {label}; checkpoint not found:', model_dir)
        continue
    tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    model = AutoModel.from_pretrained(model_dir).to(device).eval()
    metrics = evaluate_validation_retrieval(tokenizer, model, device)
    final_validation_payload[label] = {'model_dir': str(model_dir), **metrics}
    save_json(metrics, RESULTS_RUN_DIR / f'final_{label}_validation.json')
    print(label, json.dumps(metrics, indent=2))
    del model, tokenizer
    gc.collect()
    if device.startswith('cuda'):
        torch.cuda.empty_cache()

if not final_validation_payload:
    print('No saved model found yet; run the training cell first.')
else:
    save_json(final_validation_payload, RESULTS_RUN_DIR / 'final_checkpoint_validation_comparison.json')


## Evaluate Externally

Use `model_best` for the safest external-lab evaluation only after V2 beats the starting checkpoint on validation MRR:

```text
results/sapbert_full_114m_contrastive_V2/artifacts/model_best
```

Use `model_best_trained` when you specifically want to test whether the continued full-114M training helps external-lab retrieval despite not yet beating the starting checkpoint on internal validation MRR:

```text
results/sapbert_full_114m_contrastive_V2/artifacts/model_best_trained
```

In the external lab notebook, set `MODEL_DIR` to one of those paths and use a separate results folder:

```python
MODEL_DIR = REPO_DIR / 'results' / 'sapbert_full_114m_contrastive_V2' / 'artifacts' / 'model_best_trained'
RESULTS_RUN_DIR = REPO_DIR / 'results' / 'sapbert_full_114m_contrastive_V2' / 'external_lab_unique_surface_best_trained'
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / 'sapbert_full_114m_contrastive_V2_external_lab_unique_surface_best_trained'
```

Since this is a new model checkpoint, re-encode corpus/query embeddings and rebuild FAISS on the first external evaluation run.
